In [ ]:
import os
import requests
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import re

# === Configuration Section ===
# NEWS_API_KEY is your credential for the NewsAPI service. 
# date_str grabs today's date in YYYY-MM-DD format, used for directory names.
# sentiment_dir and chart_dir are folders to store results based on today's date.
NEWS_API_KEY = 'e3762f837b5d4677a8fc78db2fdc0d2f'  # Replace with your NewsAPI key
date_str = datetime.today().strftime('%Y-%m-%d')
sentiment_dir = os.path.join("sentiment", date_str)
chart_dir = os.path.join("charts", date_str)
os.makedirs(sentiment_dir, exist_ok=True)
os.makedirs(chart_dir, exist_ok=True)

# === Stock Symbol => Query Mapping ===
# This dictionary maps stock symbols (e.g. 'AAPL') to descriptive queries (e.g. 'Apple Inc').
# Example: symbol 'AAPL' will be associated with the query 'Apple Inc' for the news search.
stock_queries = {
    'AAPL': 'Apple Inc',
    'MSFT': 'Microsoft Corporation',
    'GOOGL': 'Alphabet Inc',
    'AMZN': 'Amazon.com Inc',
    'NVDA': 'NVIDIA Corporation',
    'META': 'Meta Platforms Inc',
    'TSLA': 'Tesla Inc',
    'BRK-B': 'Berkshire Hathaway Inc',
    'UNH': 'UnitedHealth Group Incorporated',
    'JPM': 'JPMorgan Chase & Co'
}

# === Load FinBERT Model for Sentiment Analysis ===
# The FinBERT model is specialized for financial texts. 
# tokenizer prepares text for the model, model handles classification, 
# and pipeline combines them into a straightforward interface.
print("Loading FinBERT model...")
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
sentiment_pipeline = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

# === Helper Function to Clean Text ===
# Removes extra spaces and line breaks, returning a tidy single-line string.
def clean_text(text):
    if not text:
        return ""
    return re.sub(r'\s+', ' ', text).strip()

# === Loop Through Each Stock to Fetch News and Analyze Sentiment ===
for symbol, query in stock_queries.items():
    print(f"\nFetching news for {symbol} using query: {query}")
    
    all_articles = []

    # Loop over the past 7 days to collect recent news articles.
    # Example: if today is 2023-01-10, it will collect from 2023-01-10, 2023-01-09, etc.
    for i in range(7):
        day = datetime.today() - timedelta(days=i)
        day_str = day.strftime('%Y-%m-%d')

        # Construct the URL to query NewsAPI for each individual day, limiting results to 14 articles.
        url = (
            f"https://newsapi.org/v2/everything?q={query}&from={day_str}&to={day_str}"
            f"&sortBy=publishedAt&pageSize=14&apiKey={NEWS_API_KEY}&language=en"
        )

        # Try getting the articles; if this fails, skip the day.
        try:
            response = requests.get(url)
            response.raise_for_status()
            articles = response.json().get("articles", [])
        except Exception as e:
            print(f"Error fetching news for {symbol} on {day_str}: {e}")
            continue

        # For each article, extract and clean title/description if a publish date is present.
        for article in articles:
            if not article.get("publishedAt"):
                continue
            title = clean_text(article.get("title", ""))
            description = clean_text(article.get("description", ""))
            if title:
                all_articles.append({
                    "date": article["publishedAt"][:10],
                    "title": title,
                    "description": description
                })

    # If no articles were gathered, move on.
    if not all_articles:
        print(f"No valid news articles found for {symbol}.")
        continue

    # Prepare all fetched text for the sentiment model, combining title and description.
    texts = [f"{a['title']}. {a['description']}" for a in all_articles]
    try:
        # The sentiment_pipeline returns labels (POSITIVE, NEGATIVE, NEUTRAL) and confidence scores.
        results = sentiment_pipeline(texts)
    except Exception as e:
        print(f"Sentiment analysis failed for {symbol}: {e}")
        continue

    # Create a DataFrame of articles and sentiment results for easier processing and saving.
    df = pd.DataFrame(all_articles)
    df["sentiment"] = [r["label"] for r in results]
    df["confidence"] = [r["score"] for r in results]

    # Save the sentiment DataFrame to a CSV file for reference.
    csv_path = os.path.join(sentiment_dir, f"{symbol}_sentiment.csv")
    df.to_csv(csv_path, index=False)
    print(f"Saved sentiment CSV to {csv_path}")

    # Create a bar chart showing the count of articles by sentiment label.
    plt.figure(figsize=(6, 4))
    df["sentiment"].value_counts().plot(kind='bar', color=["green", "red", "gray"])
    plt.title(f"Sentiment for {symbol} News (Last 7 Days, 14/Day)")
    plt.xlabel("Sentiment")
    plt.ylabel("Number of Articles")
    plt.xticks(rotation=0)
    plt.tight_layout()

    # Save the chart image to visualize sentiment distribution other than in CSV format.
    chart_path = os.path.join(chart_dir, f"{symbol}_chart.png")
    plt.savefig(chart_path)
    plt.close()
    print(f"Saved sentiment chart to {chart_path}")

# Summary messages indicating where all outputs are stored.
print(f"\n✅ All sentiment CSVs saved in: {sentiment_dir}")
print(f"✅ All sentiment charts saved in: {chart_dir}")
